In [1]:
# =======================================================================
# SCRIPT 4: SUMARIZAÇÃO HIERÁRQUICA — VARIANTE SAMSUM + QMSUM FT
# Reaproveita documento_condensado do SAMSum FT + QMSum FT na etapa final
# =======================================================================

# =======================================================================
# CÉLULA 0 — Configurações de memória
# =======================================================================
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_SKIP_ALLOCATOR_WARMUP"] = "1"

In [2]:
# =======================================================================
# CÉLULA 1 — Instalação de dependências
# =======================================================================
!pip uninstall torchvision -y
!pip install -q -U transformers datasets peft accelerate
!pip install -q nltk rouge-score
!pip install -q "torchao>=0.16.0"

Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 87.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 50.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 55.9 MB/s eta 0:00:00


In [3]:
# =======================================================================
# CÉLULA 2 — Imports
# =======================================================================
import warnings
import logging
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

import torch
import pandas as pd
import numpy as np
import nltk
import os
import shutil
from transformers import AutoTokenizer, AutoModelForCausalLM, StoppingCriteria, StoppingCriteriaList
from peft import PeftModel
from tqdm import tqdm
from google.colab import files as colab_files
import zipfile

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

True

In [4]:
# =======================================================================
# CÉLULA 3 — Configurações gerais
# =======================================================================
MODEL_ID = "microsoft/bitnet-b1.58-2B-4T-bf16"
VARIANTE = "samsum_qmsumft"
LORA_PATH = "./modelo_final_qmsum"
OUTPUT_DIR = f"./resultados_hierarquico_{VARIANTE}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MAX_TOKENS_CONDENSADO = 4096
BACKUP_A_CADA = 20

RETOMAR_DE_BACKUP = False
CAMINHO_BACKUP_RETOMADA = f"{OUTPUT_DIR}/resultados_{VARIANTE}.csv"
CAMINHO_CSV = os.path.join(OUTPUT_DIR, f"resultados_{VARIANTE}.csv")

SEED = 42

In [5]:
# =======================================================================
# CÉLULA 4 — Upload dos pesos LoRA do QMSum + CSV do SAMSum já processado
# =======================================================================
print("Faça upload do backup_qmsum_completo.zip (pesos LoRA)")
uploaded = colab_files.upload()
with zipfile.ZipFile("backup_qmsum_completo.zip", "r") as zip_ref:
    zip_ref.extractall("./")
print(f"Pesos LoRA disponíveis em: {LORA_PATH}")

print("\nFaça upload do backup_hierarquico_samsum.zip (documentos condensados já gerados)")
uploaded = colab_files.upload()
with zipfile.ZipFile("backup_hierarquico_samsum.zip", "r") as zip_ref:
    zip_ref.extractall("./")
print("CSV do SAMSum disponível!")

Faça upload do backup_qmsum_completo.zip (pesos LoRA)


Saving backup_qmsum_completo.zip to backup_qmsum_completo.zip
Pesos LoRA disponíveis em: ./modelo_final_qmsum

Faça upload do backup_hierarquico_samsum.zip (documentos condensados já gerados)


Saving backup_hierarquico_samsum.zip to backup_hierarquico_samsum.zip
CSV do SAMSum disponível!


In [6]:
# =======================================================================
# CÉLULA 5 — Carregamento do CSV já processado (SAMSum)
# =======================================================================
# Localiza o CSV automaticamente
caminho_samsum = None
for root, dirs, files in os.walk("./"):
    for f in files:
        if f == "resultados_samsum.csv":
            caminho_samsum = os.path.join(root, f)

if caminho_samsum is None:
    raise FileNotFoundError("resultados_samsum.csv não encontrado no ZIP enviado.")

print(f"CSV encontrado em: {caminho_samsum}")
df_samsum = pd.read_csv(caminho_samsum)
print(f"-> {len(df_samsum)} instâncias carregadas (documentos condensados já prontos)")
print(f"\nColunas: {df_samsum.columns.tolist()}")

CSV encontrado em: ./resultados/resultados_samsum.csv
-> 281 instâncias carregadas (documentos condensados já prontos)

Colunas: ['instancia_idx', 'query', 'resumo_referencia', 'n_chunks', 'tokens_condensado', 'documento_condensado', 'resumos_chunks', 'resumo_final']


In [7]:
# =======================================================================
# CÉLULA 6 — Funções auxiliares
# =======================================================================

class StopOnToken(StoppingCriteria):
    def __init__(self, stop_token_ids):
        self.stop_token_ids = stop_token_ids

    def __call__(self, input_ids, scores, **kwargs):
        for stop_id in self.stop_token_ids:
            if input_ids[0][-1] == stop_id:
                return True
        return False

def get_stopping_criteria(tokenizer):
    stop_sequences = ["#####", "####", "###", "## Questions", "## Question",
                      "### Meeting", "def ", "```", "Note:", "---"]
    stop_ids = []
    for seq in stop_sequences:
        ids = tokenizer.encode(seq, add_special_tokens=False)
        if ids:
            stop_ids.append(ids[0])
    return StoppingCriteriaList([StopOnToken(stop_ids)])

def limpar_output(texto, stop_sequences):
    for seq in stop_sequences:
        if seq in texto:
            texto = texto[:texto.index(seq)].strip()
    return texto

def gerar_texto(model, tokenizer, prompt, max_new_tokens,
                repetition_penalty, stopping_criteria):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096 - max_new_tokens
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=repetition_penalty,
            stopping_criteria=stopping_criteria
        )

    gerado = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    return gerado

def responder_query_qmsumft(model, tokenizer, query,
                            documento_condensado, stopping_criteria):
    """Responde a query com QMSum FT — mesmo formato do treino, com documento condensado no lugar da transcrição."""
    stop_sequences = ["#####", "####", "###", "## Questions", "## Question",
                      "### Meeting", "def ", "```", "Note:", "---"]

    prompt = (
        "### Instruction:\nWrite a concise summary answering the question "
        "based on the meeting transcript.\n\n"
        f"### Meeting History and Question:\n{query}\n\n"
        f"{documento_condensado}\n\n"
        "### Summary:\n"
    )

    gerado = gerar_texto(
        model, tokenizer, prompt,
        max_new_tokens=256,
        repetition_penalty=1.5,
        stopping_criteria=stopping_criteria
    )

    return limpar_output(gerado, stop_sequences)

def fazer_backup(output_dir, variante):
    pasta_backup = f"./backup_temp_{variante}"
    os.makedirs(pasta_backup, exist_ok=True)
    shutil.copytree(output_dir, f"{pasta_backup}/resultados",
                    dirs_exist_ok=True)
    zip_nome = f"./backup_hierarquico_{variante}.zip"
    shutil.make_archive(zip_nome.replace(".zip", ""), "zip", pasta_backup)
    print(f"\nBackup salvo: {zip_nome} "
          f"({os.path.getsize(zip_nome) / (1024**2):.1f} MB)")
    colab_files.download(zip_nome)

In [8]:
# =======================================================================
# CÉLULA 7 — Carregamento do modelo (apenas QMSum FT — etapa única)
# =======================================================================
print("Carregando tokenizador...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
stopping_criteria = get_stopping_criteria(tokenizer)

print("Carregando QMSum FT para etapa final...")
model_qmsumft = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map={"": "cuda:0"},
    attn_implementation="sdpa"
)
model_qmsumft = PeftModel.from_pretrained(model_qmsumft, LORA_PATH)
model_qmsumft.eval()
print("QMSum FT carregado!")

Carregando tokenizador...


config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/4.83G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/332 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/199 [00:00<?, ?B/s]

QMSum FT carregado!


In [9]:
# =======================================================================
# CÉLULA 7B — Retomada de backup (só rodar se RETOMAR_DE_BACKUP = True)
# =======================================================================
if RETOMAR_DE_BACKUP:
    print("Faça upload do backup para retomada:")
    uploaded = colab_files.upload()
    zip_backup = list(uploaded.keys())[0]
    with zipfile.ZipFile(zip_backup, "r") as zip_ref:
        zip_ref.extractall("./")

    for root, dirs, files in os.walk("./"):
        for f in files:
            if f == f"resultados_{VARIANTE}.csv":
                CAMINHO_BACKUP_RETOMADA = os.path.join(root, f)
                print(f"CSV encontrado em: {CAMINHO_BACKUP_RETOMADA}")

    df_progresso = pd.read_csv(CAMINHO_BACKUP_RETOMADA)
    indices_processados = set(df_progresso["instancia_idx"].tolist())
    resultados = df_progresso.to_dict("records")
    print(f"-> {len(resultados)} instâncias já processadas.")
    print(f"-> Retomando da instância {max(indices_processados)+1}...")
else:
    print("Iniciando do zero...")
    resultados = []
    indices_processados = set()

Iniciando do zero...


In [10]:
# =======================================================================
# CÉLULA 8 — Pipeline principal (apenas etapa final — reaproveita condensação)
# =======================================================================
print(f"\nIniciando pipeline — variante: {VARIANTE.upper()}")
print(f"Total de instâncias: {len(df_samsum)}")
print(f"Condensação: REAPROVEITADA do SAMSum FT | Etapa final: QMSum FT")
print(f"Backup automático a cada {BACKUP_A_CADA} instâncias\n")

for i, row in tqdm(df_samsum.iterrows(), total=len(df_samsum),
                   desc=f"Hierárquico ({VARIANTE})"):

    if i in indices_processados:
        continue

    query = row["query"]
    documento_condensado = row["documento_condensado"]
    tokens_condensado = row["tokens_condensado"]

    # Responde a query com QMSum FT, usando o documento condensado já pronto
    resumo_final = responder_query_qmsumft(
        model_qmsumft, tokenizer, query,
        documento_condensado, stopping_criteria
    )

    resultados.append({
        "instancia_idx": i,
        "query": query,
        "resumo_referencia": row["resumo_referencia"],
        "n_chunks": row["n_chunks"],
        "tokens_condensado": tokens_condensado,
        "documento_condensado": documento_condensado,
        "resumos_chunks": row["resumos_chunks"],
        "resumo_final": resumo_final
    })

    # Salva CSV progressivamente
    pd.DataFrame(resultados).to_csv(CAMINHO_CSV, index=False)

    # Backup automático a cada N instâncias
    processados_nesta_sessao = len(resultados) - len(indices_processados)
    if processados_nesta_sessao % BACKUP_A_CADA == 0:
        print(f"\n[Checkpoint] {len(resultados)} instâncias processadas")
        fazer_backup(OUTPUT_DIR, VARIANTE)

print(f"\nPipeline concluído! Total: {len(resultados)} instâncias")


Iniciando pipeline — variante: SAMSUM_QMSUMFT
Total de instâncias: 281
Condensação: REAPROVEITADA do SAMSum FT | Etapa final: QMSum FT
Backup automático a cada 20 instâncias



Hierárquico (samsum_qmsumft):   0%|          | 0/281 [00:00<?, ?it/s][transformers] Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
Hierárquico (samsum_qmsumft):   7%|▋         | 19/281 [10:59<2:30:46, 34.53s/it][transformers] Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation fo


[Checkpoint] 20 instâncias processadas

Backup salvo: ./backup_hierarquico_samsum_qmsumft.zip (0.0 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...


[Checkpoint] 280 instâncias processadas

Backup salvo: ./backup_hierarquico_samsum_qmsumft.zip (0.3 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Hierárquico (samsum_qmsumft): 100%|██████████| 281/281 [2:34:09<00:00, 32.92s/it]


Pipeline concluído! Total: 281 instâncias


In [11]:
# =======================================================================
# CÉLULA 9 — Verificação de exemplos
# =======================================================================
df = pd.read_csv(CAMINHO_CSV)
print(f"Total salvo: {len(df)} instâncias\n")

for i in range(min(3, len(df))):
    print(f"\n{'='*60}")
    print(f"INSTÂNCIA {i+1}")
    print(f"Chunks: {df.iloc[i]['n_chunks']} | "
          f"Tokens condensado: {df.iloc[i]['tokens_condensado']}")
    print(f"\nQUERY:\n{df.iloc[i]['query']}")
    print(f"\nREFERÊNCIA:\n{df.iloc[i]['resumo_referencia']}")
    print(f"\nRESUMO FINAL:\n{df.iloc[i]['resumo_final']}")

Total salvo: 281 instâncias


INSTÂNCIA 1
Chunks: 13 | Tokens condensado: 468

QUERY:
Summarize the discussion about the efficacy of the law.

REFERÊNCIA:
Barry Hughes first stated that children had fewer rights than adults and therefore the law should be enforced to defend physical assault. As such social behavior was not available now, the law should change to reflect that. The discussion then turned to talk about the legal framework and its prosecution.

RESUMO FINAL:
During this session's discussions surrounding legislative reforms concerning penalties for various transgressions including but not limited to sexual assault allegations involving minors as well as other forms like domestic violence among others; there were several key points raised by Barry Hugh who is an advocate within our system arguing strongly towards implementing measures aimed specifically targeting individuals involved directly causing harm through repeated acts resulting either physically harmful effects upon

In [12]:
# =======================================================================
# CÉLULA 10 — Backup final completo
# =======================================================================
fazer_backup(OUTPUT_DIR, VARIANTE)
print("Script concluído!")


Backup salvo: ./backup_hierarquico_samsum_qmsumft.zip (0.3 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Script concluído!
